# Studi Kasus 6 — Deteksi Awal Perubahan Tutupan Hutan Multi-Sensor
**Materi:** Remote sensing, deforestation alerts, optical + radar  
Simulasi ini memperlihatkan bagaimana ketersediaan optik yang terganggu awan dapat dipadukan dengan indikator radar untuk menghasilkan alert awal.

## Rumusan masalah dan batasan
Kita ingin menandai lokasi sintetis yang menunjukkan perubahan antara periode sebelum dan sesudah. Ini bukan analisis citra satelit asli dan tidak menentukan penyebab deforestasi. Ambang dan indikator bersifat edukatif, wajib divalidasi sebelum penggunaan nyata.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(606)
n = 160
changed = rng.binomial(1, 0.22, n)
cloudy = rng.binomial(1, 0.38, n)
# Fitur optik: indeks vegetasi sebelum/sesudah. Nilai simulasi saja.
optical_before = rng.normal(0.72, 0.06, n)
optical_after = optical_before - changed*rng.normal(0.32, 0.08, n) + rng.normal(0, 0.04, n)
optical_after = np.clip(optical_after, -0.1, 1.0)
optical_after_observed = optical_after.copy()
optical_after_observed[cloudy == 1] = np.nan
# Fitur radar sederhana: perubahan backscatter sintetis.
radar_before = rng.normal(-8.0, 1.2, n)
radar_after = radar_before + changed*rng.normal(3.0, 1.1, n) + rng.normal(0, 0.7, n)
samples = pd.DataFrame({
    "lokasi_id":[f"LOC-{i:03d}" for i in range(n)],
    "label_perubahan_simulasi":changed,
    "awan":cloudy,
    "optik_sebelum":optical_before,
    "optik_sesudah_observasi":optical_after_observed,
    "radar_sebelum":radar_before,
    "radar_sesudah":radar_after
})
samples.head()

In [ ]:
samples["delta_optik"] = samples["optik_sebelum"] - samples["optik_sesudah_observasi"]
samples["delta_radar"] = samples["radar_sesudah"] - samples["radar_sebelum"]
# Aturan demo: perubahan optik yang cukup besar ATAU perubahan radar yang cukup besar.
# Threshold tidak tervalidasi dan tidak boleh dianggap standar operasional.
optical_alert = samples["delta_optik"].fillna(0) > 0.18
radar_alert = samples["delta_radar"] > 1.8
samples["alert_awal"] = (optical_alert | radar_alert).astype(int)
samples["dasar_alert"] = np.select(
    [optical_alert & radar_alert, optical_alert, radar_alert],
    ["optik+radar", "optik", "radar"],
    default="tidak ada indikator"
)
display(samples[["lokasi_id","awan","delta_optik","delta_radar","alert_awal","dasar_alert"]].head(12))

In [ ]:
# Evaluasi hanya terhadap label simulasi, bukan ground truth lingkungan nyata.
tp = int(((samples["alert_awal"]==1) & (samples["label_perubahan_simulasi"]==1)).sum())
fp = int(((samples["alert_awal"]==1) & (samples["label_perubahan_simulasi"]==0)).sum())
fn = int(((samples["alert_awal"]==0) & (samples["label_perubahan_simulasi"]==1)).sum())
tn = int(((samples["alert_awal"]==0) & (samples["label_perubahan_simulasi"]==0)).sum())
precision = tp/(tp+fp) if tp+fp else 0
recall = tp/(tp+fn) if tp+fn else 0
print({"TP":tp, "FP":fp, "FN":fn, "TN":tn,
       "precision_simulasi":round(precision,3), "recall_simulasi":round(recall,3)})
print("Proporsi observasi optik tertutup awan:", round(samples["awan"].mean(), 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(samples.loc[samples["awan"]==0, "delta_optik"].dropna(), bins=18, alpha=0.75, label="optik tersedia")
axes[0].hist(samples.loc[samples["awan"]==1, "delta_optik"].dropna(), bins=18, alpha=0.75, label="optik tersedia saat awan=1")
axes[0].set_title("Distribusi delta optik yang tersedia")
axes[0].set_xlabel("Penurunan indikator optik")
axes[0].legend()
axes[1].scatter(samples["delta_radar"], samples["delta_optik"], c=samples["alert_awal"], alpha=0.7)
axes[1].set_xlabel("Perubahan radar sintetis")
axes[1].set_ylabel("Penurunan optik (NaN diganti 0 untuk plot)")
axes[1].set_title("Indikator perubahan dan alert")
plt.tight_layout()
plt.show()

## Interpretasi dan kehati-hatian
- Citra optik memberikan informasi vegetasi tetapi dapat terganggu awan.
- Radar dapat memberi pengamatan berbeda, tetapi interpretasinya bergantung pada struktur vegetasi, kelembapan, geometri, dan konteks sensor.
- Alert adalah indikasi perubahan, bukan bukti otomatis tentang penyebab.
- Kategori penyebab seperti pertanian, logging, atau pertambangan memerlukan konteks tambahan; validasi lapangan mungkin dibutuhkan.
- Sebelum penggunaan nyata, pakai data sensor resmi, prosedur preprocessing yang tepat, ambang tervalidasi, penilaian kesalahan, serta dokumentasi ketidakpastian.